# Lecture 16 - Quiz, and Ordinary Differential Equation Methods: Euler and Runge-Kutta

We will start with a quiz based off of the material from homework 6, and the past few weeks worth of lecture material. All students will have 10 minutes to complete the quiz, and the quiz will start promptly at 9:05 AM at the start of class. Please bring a pencil or pen for the quiz.

Next, we will begin our discussion of ODE equation integration methods.

In astrophysics, many things change with time, and can be modelled with ordinary differential equations. Some of these equations have analytic solutions, but for the vast majority of cases, it is far easier to integrate the equation numerically to understand their solultions. In the next few lectures, we will be discussing methods to numerically determine the solutions to ordinary differential equations.

### Accepting and submitting

To accept this assignment on Classroom 50 with your GitHub profile enrolled in the class, click on the link \
https://classroom50.org/PsuAstro410/410astro26/assignments/lecture-16/accept \
and follow the steps the webpage prompts.

To submit this assignment, follow the steps in "First time Classroom 50 setup" and "Accepting and Submitting Assignments" on the course webpage:\
https://psuastro410.github.io/tips/github/

### Resources and Acknowledgements

We have used material from the following sources: \
https://zingale.github.io/computational_astrophysics/ODEs/ODEs-euler.html \
https://zingale.github.io/computational_astrophysics/ODEs/ODEs-midpoint.html \
https://zingale.github.io/computational_astrophysics/ODEs/ODEs-rk4.html

We also thank Shane Davis for material that went into the design of the in-class exercises.

In [ ]:
# Nice things to have this lecture
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## Test case: orbits

We'll use the case of the Earth orbiting the Sun as our test case (we'll assume that the mass of the Earth is insignificant compared to the mass of the Sun).

Kepler's 3rd law (neglecting the orbiting object's mass) is:

$$ 4\pi^2 a^3 = G M_\star P^2$$

If we work in units of solar masses, astronomical units (AU), and years, then Kepler's third law is

$$a^3 = P^2$$

which implies that Newton's gravitational constant is $G = 4\pi^2$

We will put the Sun at the origin of coordinates.  

![circular orbit around the Sun](https://zingale.github.io/computational_astrophysics/_images/ellipse_initial_cond_e0.0.png)

Then our system of equations is

$$\dot{\bf r} = {\bf v}$$

$$\dot{\bf v} = -\frac{GM_\star {\bf r}}{r^3}$$

where ${\bf r} = (x, y)$ and ${\bf v} = (u, v)$.

**Note:**
We could have written our
orbit system as:
$$\ddot{\bf r} = - \frac{GM_\star {\bf r}}{r^3}$$
But this is a second-order ODE.  Most ODE integration methods are designed to work with systems of first-order ODEs, and we can always write a high-order ODE as a system of first-order ODEs by introducing new variables (in this case, velocity, ${\bf v}$).

## Euler's method

The simplest method to advance the solution starts with the first-order difference approximations we found for a derivative:

$$\frac{{\bf r}^{n+1} - {\bf r}^n}{\tau} = {\bf v}^n + \mathcal{O}(\tau)$$

$$\frac{{\bf v}^{n+1} - {\bf v}^n}{\tau} = {\bf a}({\bf r}^n, {\bf v}^n) + \mathcal{O}(\tau)$$

Here, $\tau$ is the timestep we take and we denote the time-level of the state using a subscript,
so $t^n$ is the current time and $t^{n+1} = t^n + \tau$.
Similarly, our notation ${\bf r}^n$ means ${\bf r}(t^{n})$.  This is called [Euler's method](https://en.wikipedia.org/wiki/Euler_method).

Expressing the new state in terms of the old:

$${\bf r}^{n+1} = {\bf r}^n + \tau \,{\bf v}^n + \mathcal{O}(\tau^2)$$

$${\bf v}^{n+1} = {\bf v}^n + \tau \,{\bf a}({\bf r}^n, {\bf v}^n) + \mathcal{O}(\tau^2)$$

Graphically, for the ODE $\dot{y} = -y$, Euler's method looks like:

![Euler's method](https://zingale.github.io/computational_astrophysics/_images/rk2_Euler.png)

We find the slope at $t^n$ and follow it forward for the full $\tau$ to $t^{n+1}$.

Now let's do our orbits.  We need a function that returns the righthand side of the ODEs, so we'll write that first.  We'll define a special type, called `OrbitState`, that holds the position and velocity, as well as defines operators that allow us to add two `OrbitState`s and multiply by a constant.

In [ ]:
class OrbitState:
    # a container to hold the star positions
    def __init__(self, x, y, u, v):
        self.x = x
        self.y = y
        self.u = u
        self.v = v

    def __add__(self, other):
        return OrbitState(self.x + other.x, self.y + other.y,
                          self.u + other.u, self.v + other.v)

    def __sub__(self, other):
        return OrbitState(self.x - other.x, self.y - other.y,
                          self.u - other.u, self.v - other.v)

    def __mul__(self, other):
        return OrbitState(other * self.x, other * self.y,
                          other * self.u, other * self.v)

    def __rmul__(self, other):
        return self.__mul__(other)

    def __str__(self):
        return f"{self.x:10.6f} {self.y:10.6f} {self.u:10.6f} {self.v:10.6f}"

We'll write our righthand side function in terms of the `OrbitState`.  It will return a new `OrbitState` representing the derivatives.

In [ ]:
GM = 4*np.pi**2

def rhs(state):
    """ RHS of the equations of motion."""

    # current radius
    r = np.sqrt(state.x**2 + state.y**2)

    # position
    xdot = state.u
    ydot = state.v

    # velocity
    udot = -GM * state.x / r**3
    vdot = -GM * state.y / r**3

    return OrbitState(xdot, ydot, udot, vdot)

Many times and ODE solver will just give you the solution at the end of integration, so we'll store a list of `OrbitState` objects that constitute the history.

Now we can write a driver that takes the initial conditions and integrates to a time $T$ using a constant timestep $\tau$.

In [ ]:
def euler_orbit(state0, tau, T):
    """integrate an orbit given an initial position, pos0, and velocity, vel0,
    using first-order Euler integration"""
    
    times = []
    history = []
    
    # initialize time
    t = 0
    
    # store the initial conditions
    times.append(t)
    history.append(state0)
    
    # main timestep loop
    while t < T:
        
        state_old = history[-1]
        
        # make sure that the last step does not take us past T
        tau = min(tau, T - t)

        # get the RHS
        ydot = rhs(state_old)
        
        # do the Euler update
        state_new = state_old + tau * ydot
        t += tau
        
        # store the state
        times.append(t)
        history.append(state_new)
        
    return times, history

We'll start out on the +y axis and give the Earth a counterclockwise circular velocity.

In [ ]:
x0 = 0
y0 = 1
u0 = -np.sqrt(GM/y0)
v0 = 0

state0 = OrbitState(x0, y0, u0, v0)

## First orbit

Now let's integrate for one period (T = 1, since our time units are years).  We'll our timestep to be 1 month&mdash;let's see how we do with that.

In [ ]:
T = 1
tau = T/12.0

times, history = euler_orbit(state0, tau, 1)

Let's plot the solution.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

xs = [q.x for q in history]
ys = [q.y for q in history]

# draw the Sun
ax.scatter([0], [0], marker=(20,1), color="y", s=250)

# draw the orbit
ax.plot(xs, ys)

ax.set_aspect("equal")
ax.set_xlabel("x [AU]")
ax.set_ylabel("y [AU]")

plt.show()

The solution looks awful.  It should be a circle, and after 1 year, we should be right back at where we started.

## Varying the timestep

Let's look at how we converge with the size of the timestep, $\tau$.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

# draw the Sun
ax.scatter([0], [0], marker=(20,1), color="y", s=250)

taus = [0.1, 0.01, 0.001]

for tau in taus:
    times, history = euler_orbit(state0, tau, 1)
    xs = [q.x for q in history]
    ys = [q.y for q in history]
    ax.plot(xs, ys, label=rf"$\tau = {tau:6.4f}$")

ax.set_aspect("equal")
ax.legend()
ax.set_xlabel("x [AU]")
ax.set_ylabel("y [AU]")

plt.show()

Clearly our solution gets better as we reduce the timestep, but we need to reduce it a lot.  At the top we said that the Euler method is globally first-order accurate.  Let's try to confirm that and then let's look at how we can do higher-order integration.

## Measuring convergence

Imagine evolving for a single period and then asking how well did the integrator do.  We need some metric to determine this.  We can imagine a few different measures:

1. compare the current distance to the Sun to the original distance to the Sun

2. compute the distance from the current $(x, y)$ position to the starting position

3. compute the total energy at the end of the integration and compare to the initial total energy.

There are likely many more.  Unlike the integrals we did previously, where the natural error to compute was the difference between the numerical integral and the analytic one, here there is not a single obvious metric.  So we should pick one or more and look for convergence.  Depending on what our goal is, one metric might be more meaningful than another.

Let's define an error function based on metric 1 above

In [ ]:
def error(history):
    
    # define the error to be distance from (0, 0) at end compared to start
    R_orig = np.sqrt(history[0].x**2 + history[0].y**2)
    R_new = np.sqrt(history[-1].x**2 + history[-1].y**2)
    e = np.abs(R_new - R_orig)
    
    return e

Now let's integrate a bunch of orbits, reducing $\tau$ by 2 each time.  Since this is first order accurate, the error should decrease by a factor of 2.

In [ ]:
taus = [0.1, 0.05, 0.025, 0.0125, 0.00625]

for tau in taus:
    times, history = euler_orbit(state0, tau, 1)
    print(f"{tau:8} : {error(history)}")

We see that the decrease in the error is essentially 2.  It is certainly not better than first-order accurate.

# 2nd Order Runge-Kutta / Midpoint Method

We'll continue to work on the orbit problem.  

The Euler method was based on a first-order difference approximation to the derivative.  But we know that a centered-derivative is second order accurate, so we can try to update our system in the form:

$$\frac{{\bf r}^{n+1} - {\bf r}^n}{\tau} = {\bf v}^{n+1/2} + \mathcal{O}(\tau^2)$$

$$\frac{{\bf v}^{n+1} - {\bf v}^n}{\tau} = {\bf a}^{n+1/2} + \mathcal{O}(\tau^2)$$

Then the updates are:

$${\bf r}^{n+1} = {\bf r}^n + \tau \, {\bf v}^{n+1/2} + \mathcal{O}(\tau^3)$$

$${\bf v}^{n+1} = {\bf v}^n + \tau \, {\bf a}^{n+1/2} + \mathcal{O}(\tau^3)$$


This is locally third-order accurate (but globally second-order accurate), but we don't know how to compute the state at the half-time.

To find the $n+1/2$ state, we first use Euler's method to predict the state at the midpoint in time.  We then use this provisional state to evaluate the accelerations at the midpoint and use those to update the state fully through $\tau$.  

The two step process appears as:

$${\bf r}^\star = {\bf r}^n + (\tau / 2) {\bf v}^n$$

$${\bf v}^\star = {\bf v}^n + (\tau / 2) {\bf a}^n$$

then we use this for the full update:

$${\bf r}^{n+1} = {\bf r}^n + \tau \,  {\bf v}^\star$$

$${\bf v}^{n+1} = {\bf v}^n + \tau \, {\bf a}({\bf r}^\star)$$

This is called the [midpoint method](https://en.wikipedia.org/wiki/Midpoint_method) or 2nd-order Euler's method.

Graphically this looks like the following:

**First step:**
First we take a half step and we evaluate the slope at the midpoint:

![RK2 half step](https://zingale.github.io/computational_astrophysics/_images/rk2_halfEuler.png)

**Final update:**
Then we go back to $t^n$ but follow the slope we found above all the way to $t^{n+1}$:

![RK2 final](https://zingale.github.io/computational_astrophysics/_images/rk2_final.png)

Notice how the final step (the red line) is parallel to the slope we computed at $t^{n+1/2}$.  Also note that the solution at $t^{n+1}$ is much closer to the analytic solution than in the figure from Euler's method.

## Implementation

Let's see how this method does with the orbit problem.

In [ ]:
def int_rk2(state0, tau, T):

    times = []
    history = []
    
    # initialize time
    t = 0
    
    # store the initial conditions
    times.append(t)
    history.append(state0)
    
    # main timestep loop
    while t < T:
        
        state_old = history[-1]
        
        # make sure that the last step does not take us past T
        tau = min(tau, T - t)

        # get the RHS
        ydot = rhs(state_old)
    
        # predict the state at the midpoint
        state_tmp = state_old + 0.5 * tau * ydot
        
        # evaluate the RHS at the midpoint
        ydot = rhs(state_tmp)
        
        # do the final update
        state_new = state_old + tau * ydot
        t += tau
        
        # store the state
        times.append(t)
        history.append(state_new)
        
    return times, history

**Note:**
Our `int_rk2()` function is almost identical to our first-order Euler implementation.
There are actually only 2 lines that are different:

```python
        # predict the state at the midpoint
        state_tmp = state_old + 0.5 * tau * ydot
        
        # evaluate the RHS at the midpoint
        ydot = rhs(state_tmp)
```

## Testing

Integrate our orbit.

In [ ]:
T = 1
tau = T/12.0

def initial_conditions():
    x0 = 0
    y0 = 1
    u0 = -np.sqrt(GM / y0)
    v0 = 0

    return OrbitState(x0, y0, u0, v0)

state0 = initial_conditions()

times, history = int_rk2(state0, tau, 1)

Let's plot our orbit. We first define a fancy plotting function

In [ ]:
def plot_func(history, ax=None, label=None):
    """make a plot of the solution.  If ax is None we setup a figure
    and make the entire plot returning the figure object, otherwise, we
    just append the plot to a current axis"""

    fig = None

    if ax is None:
        fig = plt.figure()
        ax = fig.add_subplot(111)

        # draw the Sun
        ax.scatter([0], [0], marker=(20,1), color="y", s=250)

    # draw the orbit
    xs = [q.x for q in history]
    ys = [q.y for q in history]

    ax.plot(xs, ys, label=label)

    if fig is not None:
        ax.set_aspect("equal")
        ax.set_xlabel("x [AU]")
        ax.set_ylabel("y [AU]")

    return fig

Then plot the orbit

In [ ]:
fig = plot_func(history)
plt.show()

This is substantially better than the first-order Euler method.  Now let's look at a range of timesteps.

In [ ]:
taus = [0.1, 0.05, 0.025, 0.0125]

for n, tau in enumerate(taus):
    times, history = int_rk2(state0, tau, 1)
    
    if n == 0:
        fig = plot_func(history, label=rf"$\tau = {tau:6.4f}$")
    else:
        plot_func(history, ax=fig.gca(), label=rf"$\tau = {tau:6.4f}$")
        
fig.gca().legend()
plt.show()

## Convergence

How does the error converge? We first define error calculation functions

In [ ]:
def error_radius(history):

    # define the error to be distance from (0, 0) at end compared to start
    R_orig = np.sqrt(history[0].x**2 + history[0].y**2)
    R_new = np.sqrt(history[-1].x**2 + history[-1].y**2)
    e = np.abs(R_new - R_orig)

    return e

def error_position(history):
    """return the difference in the distance from the Sun"""
    dx = history[0].x - history[-1].x
    dy = history[0].y - history[-1].y
    return np.sqrt(dx**2 + dy**2)

then plot the error with time

In [ ]:
for tau in [0.1, 0.05, 0.025, 0.0125, 0.00625]:
    times, history = int_rk2(state0, tau, 1)
    print(f"{tau:8} : {error_radius(history):10.5g} {error_position(history):10.5g}")

Notice that once we get past the first, very coarse $\tau$, the errors seem to decrease by a factor of 4 when we halve the timestep&mdash;as we'd expect for a 2nd order accurate method. (Actually, it looks like the measure of radius converges better than position).

# 4th-order Runge-Kutta

Now we consider a 4th order accurate method&mdash;[4th order Runge-Kutta](https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods#The_Runge%E2%80%93Kutta_method) (RK4).  For many problems, this method will work very well and as a result, RK4 is widely used.

We'll consider a general system of first order differential equations:

$$\dot{\bf y} = {\bf f}(t, {\bf y})$$

**Step 1:** Compute k1

The advance begins by estimating the derivatives (righthand side or slope) at time $t^n$.  We'll call this ${\bf k}_1$.

$${\bf k}_1 = {\bf f}(t^n, {\bf y}^n)$$

![4th order Runge-Kutta k1](https://zingale.github.io/computational_astrophysics/_images/rk4_k1.png)

**Step 2:** Compute k2

We then follow the slope ${\bf k}_1$ to the midpoint in time, $t^{n+1/2}$ and evaluate the slope there.  We call the new slope ${\bf k}_2$.

$${\bf k}_2 = {\bf f}(t^n + \tau/2, {\bf y}^n + (\tau/2) {\bf k}_1)$$

![4th order Runge-Kutta k2](https://zingale.github.io/computational_astrophysics/_images/rk4_k2.png)

**Step 3:** Compute k3

We then go back to the start, but this time follow the new slope,
${\bf k}_2$ to the midpoint in time, $t^{n+1/2}$.  We again evaluate the slope here, and call it ${\bf k}_3$.

$${\bf k}_3 = {\bf f}(t^n + \tau/2, {\bf y}^n + (\tau/2) {\bf k}_2)$$

![4th order Runge-Kutta k3](https://zingale.github.io/computational_astrophysics/_images/rk4_k3.png)

**Step 4:** Compute k4

Finally, we go back to the start and follow ${\bf k}_3$ for the full
timestep, to $t^{n+1}$ and evaluate the slope there, calling it ${\bf k}_4$.

$${\bf k}_4 = {\bf f}(t^n + \tau, {\bf y}^n + \tau {\bf k}_3)$$

![4th order Runge-Kutta k4](https://zingale.github.io/computational_astrophysics/_images/rk4_k4.png)

**Step 5:** Final update

We then get the updated solution using a linear combination of the 4 slopes:

$${\bf y}^{n+1} = {\bf y}^n + \frac{\tau}{6} ({\bf k}_1 + 2 {\bf k}_2 + 2 {\bf k}_3 + {\bf k}_4)$$

![4th order Runge-Kutta final](https://zingale.github.io/computational_astrophysics/_images/rk4_final.png)

## RK4 Implementation

Here's our implementation.  Again, it follows the same flow as the other methods, but now with a few more calls to the `rhs()`
function.

In [ ]:
def int_rk4(state0, tau, T):

    times = []
    history = []
    
    # initialize time
    t = 0
    
    # store the initial conditions
    times.append(t)
    history.append(state0)
    
    # main timestep loop
    while t < T:
        
        state_old = history[-1]
        
        # make sure that the last step does not take us past T
        tau = min(tau, T - t)

        # get the RHS
        k1 = rhs(state_old)
         
        state_tmp = state_old + 0.5 * tau * k1
        k2 = rhs(state_tmp)
        
        state_tmp = state_old + 0.5 * tau * k2
        k3 = rhs(state_tmp)
        
        state_tmp = state_old + tau * k3
        k4 = rhs(state_tmp)
        
        # do the final update
        state_new = state_old + tau / 6.0 * (k1 + 2*k2 + 2*k3 + k4)
        t += tau
        
        # store the state
        times.append(t)
        history.append(state_new)
        
    return times, history

Let's try it out now.

In [ ]:
state0 = initial_conditions()

tau = 1.0/12.0

times, history = int_rk4(state0, tau, 1)

fig = plot_func(history)
plt.show()

This already looks pretty good, even for this very coarse timestep.

## Convergence

We can look at the convergence of RK4

In [ ]:
taus = [0.1, 0.05, 0.025]

for n, tau in enumerate(taus):
    times, history = int_rk4(state0, tau, 1)
    
    if n == 0:
        fig = plot_func(history, label=rf"$\tau = {tau:6.4f}$")
    else:
        plot_func(history, ax=fig.gca(), label=rf"$\tau = {tau:6.4f}$")

fig.gca().legend()
plt.show()

In [ ]:
for tau in [0.1, 0.05, 0.025, 0.0125, 0.00625]:
    times, history = int_rk4(state0, tau, 1)
    print(f"{tau:8} : {error_radius(history):10.5g} {error_position(history):10.5g}")

This is clearly converging faster than 2nd order.  4th order means that as we cut the timestep in half, the error should go down by $2^4$ or 16.

Beyond just converging faster, also compare the error here to the Euler and midpoint methods.  
For a timestep of $\tau = 0.025$, we are > 100 times more accurate than the midpoint method and
almost 100,000 times more accurate than Euler's method.

So despite having to call the `rhs()` function more often, we are *much more* accurate than
those other methods.

In the exercises below, we will write our algorithm in a more compact form. These exercises walk through how to write rk2 as functions, and in the homework assignment, you will write an algorithm for rk4 yourselves.

#  Exercises (10 pt)

## Muddiest moment (2 pt)

What was the most confusing part of this lecture?

**Answer here**

## Exercise 1 (8 pt) - Integrating ODEs with a function, not a class

As an example, lets consider the equation 
$$
\frac {dy}{dt} = \cos(t).
$$
We can see that we can solve this analytically by noting
$$
y(t) = \int\cos(t) dt = \sin(t) + C,
$$
where $C$ is a constant.  Let see how we can do this numerically. First we define a function for the derivatives.

Consider the following general expression for a first ODE.
$$
\frac {d\vec{y}}{dt} = \vec{f}(\vec{y}, t),
$$
where $\vec{y}$ is a vector of variables, $t$ is the independent variable, and $\vec{f}()$ is some arbitrary vector function of $\vec{y}$ and $t$.

As we just discussed, **Euler's method** uses value at $\vec{y}(t_n)$ to solve for $\vec{y}(t_{n+1})$ to be
$$
\vec{y}_{n+1} = \vec{y}_n + \vec{f}(\vec{y}_n, t_n)\Delta t.\qquad (3)
$$
If we take $\vec{f}$ to be a scalar function of just $t$, then it just becomes an ordinary integral, similar to what we have solved previously.

Lets implement this method with a more general set of functions (that will be useful for the next homework assignment). First define derivatives (the vector function $\vec{f}(\dots)$):

In [ ]:
# implement the derivative above
def derivatives(t, y): 
    return np.cos(t)

Now you might wonder why we passed y in the derivatives function. We shall return to this shortly. Now let us define Euler method.  
$$
y(t+\Delta t) = y(t) + \frac{dy}{dt}\Delta t
$$

The below code implements the Euler's method as using $dy/dt$ using `derivatives()`.

In [ ]:
# Euler's method
def euler(derivatives, t, delta_t, y): 
    return y + derivatives(t, y) * delta_t

Note that the euler function above follows the exact definition of euler's method as defined above.  Now lets drive this between $t=0$ and $t=100$.

In [ ]:
def run_euler(N=1000, y0=0, t0=0, t1=100) :
    
    delta_t = (t1-t0)/N
    y = y0
    t = t0
    
    t_array = []
    y_array = []
    for i in range(N):
        t += delta_t
        y = euler(derivatives, t, delta_t, y)
        t_array.append(t)
        y_array.append(y)
    return np.array(t_array), np.array(y_array)

Great.  Now lets plot this

In [ ]:
t_array, y_array = run_euler(N=1000)
plt.plot(t_array, y_array, '.', label='Euler')
plt.plot(t_array, np.sin(t_array), label='true solution')
plt.legend()

print("Average Error: ",np.average(abs(y_array-np.sin(t_array))))
plt.show()

## Runge-Kutta 2 step (RK2) / Midpoint method

By taylor expanding $\vec{y}(t_n + \Delta t)$, one can show that for RK2, the next step can always be written as
$$
\vec{k}_1 = \vec{f}(\vec{y}_n, t_n) \Delta t 
$$
$$
\vec{k}_2 =  \vec{f}(\vec{y}_n + \beta \vec{k}_1, t_n + \alpha \Delta t) \Delta t
$$
$$
\vec{y}_{n+1} = \vec{y}_n + a \vec{k}_1 + b \vec{k}_2
$$
When the "correction" to the method (where the slope is evalulated to give a correct estimate of the derivative) is the midpoint ( $\alpha = \beta = 1/2$, timestep $\approx \Delta t/2$), one can show that
$$
b = 1, \qquad a = 0
$$

## Exercise 1(a) - Write rk2 algorithm that updates new step (4 pt)

Write an algorithm that implements rk2 using the derivatives as defined above

In [ ]:
# implement rk2.  derivatives have the signature derivatives(t,y) as above
def rk2(derivatives, t, dt, y):
    k1 = # Expression for k1 in terms of derivatives
    k2 = # Expression for k2 in terms of derivatives
    return # Return new step

## Exercise 1(b) - Calculate solution for $t$ and $y$ (2 pt)

Finish the algorithm that runs the RK2 algorithm

In [ ]:
# code for running RK2
def run_rk2(N=1000, t0=0, t1=100):
    dt = (t1-t0)/N
    y = 0
    t = t0
    y_array = [y]
    t_array = [t]

    for i in range(N):
        ## PUT IN CODE in line below
        y = # Update to y
        t = # Update to t
        t_array.append(t)
        y_array.append(y)
    return np.array(t_array), np.array(y_array)

## Exercise 1(c) - Calculate and the solution (2 pt)

Calculate the solution numerically, and compare the numerical solution to the true solution

In [ ]:
t_array, y_array = # Put code here
plt.plot(t_array, y_array, label='RK2')
plt.plot(t_array, np.sin(t_array),'.',label='true solution')
plt.legend()

print("Average Error: ",np.average(abs(y_array-np.sin(t_array))))

plt.show()